In [1]:
import multiprocessing
from pathlib import Path
import torch
from ultralytics import YOLO

def main():
    multiprocessing.freeze_support()
    base_dir = Path(r"D:\Journal-Project\results_v6\A1")
    base_dir.mkdir(parents=True, exist_ok=True)

    data_yaml = r"D:\Journal-Project\Datasets\Q1_Harmonized_Benchmark\marine_litter\marine_litter.yaml"

    print("--- Starting A1: Controlled Baseline Training (YOLO11n, MuSGD, lr0=0.01, seed=42) ---")
    model = YOLO("yolo11n.pt")

    results = model.train(
        data=data_yaml,
        epochs=100,
        imgsz=640,
        batch=16,
        optimizer="MuSGD",
        lr0=0.01,
        seed=42,
        deterministic=True,
        amp=True,
        workers=4,
        project=str(base_dir),
        name="train_yolo11n_musgd",
        exist_ok=True,
        val=False,
        verbose=True,
    )
    print("A1 Training completed successfully!")

if __name__ == "__main__":
    main()


--- Starting A1: Controlled Baseline Training (YOLO11n, MuSGD, lr0=0.01, seed=42) ---
New https://pypi.org/project/ultralytics/8.4.211 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.182  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Journal-Project\Datasets\Q1_Harmonized_Benchmark\marine_litter\marine_litter.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hs

In [1]:
import multiprocessing
import sys, os
from pathlib import Path
import torch
from ultralytics import YOLO

# Register custom module
sys.path.insert(0, r"D:\Journal-Project")
import ultralytics.nn.modules as ul_modules
import ultralytics.nn.tasks as ul_tasks
from fa_evyolo_modules import SpectralAmplitudeNorm2d

setattr(ul_modules, "SpectralAmplitudeNorm2d", SpectralAmplitudeNorm2d)
setattr(ul_tasks, "SpectralAmplitudeNorm2d", SpectralAmplitudeNorm2d)
ul_tasks.__dict__["SpectralAmplitudeNorm2d"] = SpectralAmplitudeNorm2d
if hasattr(ul_tasks, "base_modules"):
    ul_tasks.base_modules = tuple(
        m for m in ul_tasks.base_modules if m.__name__ != "SpectralAmplitudeNorm2d"
    ) + (SpectralAmplitudeNorm2d,)

A2_DIR = Path(r"D:\Journal-Project\results_v6\A2")
A2_DIR.mkdir(parents=True, exist_ok=True)
DATA_YAML = r"D:\Journal-Project\Datasets\Q1_Harmonized_Benchmark\marine_litter\marine_litter.yaml"

def build_head_remapped_fa_evyolo():
    """
    Builds FA-EvYOLO, transfers yolo11n.pt weights,
    and remaps matching head tensors from model.23.* to model.26.*.
    """
    base = YOLO("yolo11n.pt")
    base_sd = base.model.state_dict()
    base_sum = float(base_sd["model.23.cv2.0.0.conv.weight"].sum())

    fa = YOLO("fa_evyolo.yaml")
    fa.load("yolo11n.pt")

    copied = 0
    for k, v in base_sd.items():
        if "model.23." in k:
            k26 = k.replace("model.23.", "model.26.")
            if k26 in fa.model.state_dict() and fa.model.state_dict()[k26].shape == v.shape:
                fa.model.state_dict()[k26].copy_(v)
                copied += 1

    print(f"[A2 Head Remap] Copied {copied} matching head tensors from yolo11n model.23.* to FA-EvYOLO model.26.*")

    def verify_callback(trainer):
        t_sum = float(trainer.model.state_dict()["model.26.cv2.0.0.conv.weight"].sum())
        match = abs(t_sum - base_sum) < 1e-5
        print(f"\n=======================================================")
        print(f"[VERIFY AT EPOCH 0 INSIDE TRAINER]")
        print(f"  yolo11n.pt model.23.cv2.0.0 weight sum: {base_sum:.6f}")
        print(f"  trainer    model.26.cv2.0.0 weight sum: {t_sum:.6f}")
        print(f"  EXACT CHECKSUM MATCH: {match}")
        print(f"=======================================================\n")
        assert match, "Checksum mismatch: head remapping was not preserved inside trainer!"

    fa.add_callback("on_train_start", verify_callback)
    return fa

def train_fa_evyolo_seed(seed):
    save_name = f"train_fa_evyolo_remapped_seed{seed}"
    ckpt_path = A2_DIR / save_name / "weights" / "best.pt"
    if ckpt_path.exists():
        print(f"Skipping {save_name}, already exists at {ckpt_path}")
        return

    print(f"\n>>> Starting FA-EvYOLO (Head Remapped) training with seed={seed} <<<")
    model = build_head_remapped_fa_evyolo()
    model.train(
        data=DATA_YAML,
        epochs=100,
        imgsz=640,
        batch=16,
        optimizer="MuSGD",
        lr0=0.01,
        seed=seed,
        deterministic=True,
        amp=True,
        workers=4,
        project=str(A2_DIR),
        name=save_name,
        exist_ok=True,
        val=False,
        verbose=True,
    )
    print(f">>> Completed FA-EvYOLO seed={seed} <<<\n")

def train_yolo11n_musgd_seed(seed):
    save_name = f"train_yolo11n_musgd_seed{seed}"
    ckpt_path = A2_DIR / save_name / "weights" / "best.pt"
    if ckpt_path.exists():
        print(f"Skipping {save_name}, already exists at {ckpt_path}")
        return

    print(f"\n>>> Starting YOLO11n (MuSGD) training with seed={seed} <<<")
    model = YOLO("yolo11n.pt")
    model.train(
        data=DATA_YAML,
        epochs=100,
        imgsz=640,
        batch=16,
        optimizer="MuSGD",
        lr0=0.01,
        seed=seed,
        deterministic=True,
        amp=True,
        workers=4,
        project=str(A2_DIR),
        name=save_name,
        exist_ok=True,
        val=False,
        verbose=True,
    )
    print(f">>> Completed YOLO11n (MuSGD) seed={seed} <<<\n")

def main():
    multiprocessing.freeze_support()
    print("=================================================================")
    print("              STARTING A2 MULTI-SEED TRAINING QUEUE              ")
    print("  Runs: FA-EvYOLO (remapped) seeds 42, 0, 1 | YOLO11n seeds 0, 1 ")
    print("=================================================================\n")

    # 1. FA-EvYOLO with Head Remapped: seeds 42, 0, 1
    for seed in [42, 0, 1]:
        train_fa_evyolo_seed(seed)

    # 2. YOLO11n MuSGD: seeds 0, 1 (seed 42 is already trained in A1)
    for seed in [0, 1]:
        train_yolo11n_musgd_seed(seed)

    print("\nAll 5 A2 training runs completed successfully!")

if __name__ == "__main__":
    main()


              STARTING A2 MULTI-SEED TRAINING QUEUE              
  Runs: FA-EvYOLO (remapped) seeds 42, 0, 1 | YOLO11n seeds 0, 1 


>>> Starting FA-EvYOLO (Head Remapped) training with seed=42 <<<
WARNING no model scale passed. Assuming scale='n'.
Transferred 378/547 items from pretrained weights
[A2 Head Remap] Copied 115 matching head tensors from yolo11n model.23.* to FA-EvYOLO model.26.*
New https://pypi.org/project/ultralytics/8.4.211 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.182  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Journal-Project\Datasets\Q1_Harmonized_Benchmark\marine_litt